In [ ]:
import hashlib
import shutil
from pathlib import Path

import pandas as pd

In [ ]:
def locate_repo():
    current = Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / "paper_submissions_manifest.csv").exists():
            return candidate
    for candidate in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if candidate.exists():
            for path in candidate.glob("*/paper_submissions_manifest.csv"):
                return path.parent
    raise FileNotFoundError("paper_submissions_manifest.csv")

repo = locate_repo()
manifest = pd.read_csv(repo / "paper_submissions_manifest.csv", dtype=str).fillna("")

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1048576), b""):
            h.update(chunk)
    return h.hexdigest()

paper_dir = repo / "paper_submissions"
paper_dir.mkdir(parents=True, exist_ok=True)

for row in manifest.to_dict("records"):
    source_name = row.get("input_csv", "")
    if not source_name:
        continue
    source = repo / source_name
    if not source.exists():
        raise FileNotFoundError(source)
    destination = paper_dir / f"{row['slug']}.csv"
    shutil.copyfile(source, destination)
    data = pd.read_csv(destination)
    if list(data.columns) != ["image_id", "cluster"]:
        raise ValueError(f"invalid columns: {row['slug']}")
    expected_rows = row.get("rows", "")
    if expected_rows and len(data) != int(expected_rows):
        raise ValueError(f"row count mismatch: {row['slug']}")
    expected_hash = row.get("sha256", "")
    actual_hash = file_sha256(destination)
    if expected_hash and actual_hash != expected_hash:
        raise ValueError(f"checksum mismatch: {row['slug']}")

selected = paper_dir / "salamander_eva02_aliked_local_link_refinement.csv"
submission = repo / "submission.csv"
shutil.copyfile(selected, submission)

print(f"submission: {submission}")
print(f"sha256: {file_sha256(submission)}")